In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

import numpy as np
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import mlflow

### Dataset

In [3]:
class TextDataset(Dataset):
    def __init__(
      self,
      base_path: str,
      token_dtype=np.uint16,
      return_tensor: bool = False,
    ):
        bin_path = f"{base_path}.bin"
        idx_path = f"{base_path}.idx"
        self.return_tensor = return_tensor
        self.offsets = np.fromfile(idx_path, dtype=np.int64)
        self.tokens = np.memmap(bin_path, dtype=token_dtype, mode="r")
        self._len = len(self.offsets) - 1
    def __len__(self) -> int:
        return self._len
    def __getitem__(self, index: int) -> list[int] | torch.Tensor:
        start = self.offsets[index]
        end = self.offsets[index + 1]
        seq = self.tokens[start:end]
        if self.return_tensor:
            return torch.from_numpy(seq.astype(np.uint16))
        return seq.astype(np.uint16).tolist()

In [4]:
class LMCollator:
    def __init__(self, pad_id: int, max_seq_len: int):
        self.pad_id = pad_id
        self.max_seq_len = max_seq_len
    def __call__(self, batch: list[list[int]]) -> dict[str, torch.Tensor]:
        truncated_batch = []
        for seq in batch:
            if len(seq) > self.max_seq_len:
                seq = seq[:self.max_seq_len]
            truncated_batch.append(seq)
        batch_max_len = max(max(len(s) for s in truncated_batch), 2)

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []
        for seq in truncated_batch:
            seq_len = len(seq)
            pad_len = batch_max_len - seq_len
            item_tensor = torch.tensor(seq, dtype=torch.long)

            if pad_len > 0:
                padding = torch.full((pad_len,), self.pad_id, dtype=torch.long)
                padded = torch.cat([item_tensor, padding], dim=0)
                mask = torch.cat([
                    torch.ones(seq_len, dtype=torch.bool),
                    torch.zeros(pad_len, dtype=torch.bool)
                ])
            else:
                padded = item_tensor
                mask = torch.ones(batch_max_len, dtype=torch.bool)
            inp = padded[:-1]
            tgt = padded[1:].type(torch.long).clone()
            attn = mask[:-1]

            # Mask padding positions for CrossEntropyLoss
            tgt[tgt == self.pad_id] = -100

            batch_input_ids.append(inp)
            batch_attention_mask.append(attn)
            batch_labels.append(tgt)

        return {
            "input_ids": torch.stack(batch_input_ids),
            "attention_mask": torch.stack(batch_attention_mask),
            "labels": torch.stack(batch_labels)
        }

In [5]:
from core.tokenizer import BPE
from core.config import Config

configs = Config.from_json(path="exp_1/exp_1_config.json")
tokenizer = BPE.load('exp_1/tokenizer','peak_base_1')
td_train = TextDataset("exp_1/data/train_1",return_tensor=False)
td_validation = TextDataset("exp_1/data/val",return_tensor=False)
collator = LMCollator(pad_id=tokenizer.special_tokens["<PAD>"], max_seq_len=configs.model.ctx_length)
train_loader = DataLoader(
    dataset=td_train,
    batch_size=configs.training.batch_size,
    shuffle=True,
    collate_fn=collator,
    pin_memory=True
)
val_loader = DataLoader(
    dataset=td_validation,
    batch_size=configs.training.batch_size,
    shuffle=False,
    collate_fn=collator,
    pin_memory=True
)

In [6]:
# sanity check
tokenizer.decode(next(iter(train_loader))['input_ids'][0].tolist(),False)

'Once upon a time there were two little brothers. One brother loved to cook, while the other brother loved to take pictures with his camera. The brothers often went out together on weekends to explore the city.\n\nOne weekend the brothers decided to try something new. Instead of eating takeaway, the brother who loved to cook decided to make a delicious lunch for both of them. He chopped, stirred and fried the ingredients until a pretty and unique meal was ready.\n\nThe brother who loved to take pictures took out his camera to take a photo of the tasty dish. They looked at it and both smiled, realizing the meal was just as special and unique as their friendship.\n\nThat day, the brothers learned that sometimes different things can come together and create something amazing. They decided to share their special moments and create something unique whenever they could.'

### Training

In [7]:
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import SequentialLR,LinearLR,CosineAnnealingLR
from accelerate import Accelerator

In [8]:
from pathlib import Path
import math
from tqdm.auto import tqdm
from core.config import Config

class Trainer:
    def __init__(self,
                 model: nn.Module,
                 config : Config,
                 train_loader: DataLoader,
                 val_loader: DataLoader | None = None,
                 token_seen_limit : int | None = None,
                 checkpoint_save_dir : str | None = None,
                 checkpoint_load_dir : str | None = None,
    ):
        # Configs
        self.config = config
        # Loss Function
        self.loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
        # Declare optimizer,
        self.optimizer = AdamW(
            model.parameters(),
            lr=config.training.learning_rate,
            weight_decay=config.training.weight_decay,
        )
        # LR Scheduler
        self.total_steps = math.ceil((len(train_loader) * config.training.epochs) / config.training.gradient_accumulation_steps)
        self.warmup_steps = max(1,int(self.total_steps * self.config.training.warmup_ratio))
        self.min_learning_rate = config.training.learning_rate / 10
        self.scheduler = SequentialLR(
            self.optimizer,
            schedulers=[
                LinearLR(
                    self.optimizer,
                    start_factor=0.01,
                    end_factor=1.0,
                    total_iters=self.warmup_steps,
                ),
                CosineAnnealingLR(
                    self.optimizer,
                    T_max=self.total_steps - self.warmup_steps,
                    eta_min=self.min_learning_rate,
                ),
            ],
            milestones=[self.warmup_steps],
        )
        # Trackers (for tracking training checkpoints)
        self.epochs = 0
        self.global_step = 0
        self.validate_every_step = 50
        self.best_val_loss = float("inf")
        self.token_seen_limit = token_seen_limit
        # Set Up
        self.accelerator = Accelerator(mixed_precision="fp16", gradient_accumulation_steps=config.training.gradient_accumulation_steps,)
        self.model,self.optimizer,self.scheduler,self.train_loader,self.val_loader = self.accelerator.prepare(model,self.optimizer,self.scheduler,train_loader,val_loader)
        print(f"Using {self.accelerator.device}")
        # Save checkpoint Directory
        self.checkpoint_save_dir = checkpoint_save_dir if checkpoint_save_dir else "."
        # Load checkpoint weights if available
        if checkpoint_load_dir:
            self.__load_checkpoint(checkpoint_load_dir,"best")
        print("Total Steps: ",self.total_steps)
        print("Validate After Every Step: ",self.validate_every_step)
        print("Warmup Steps : ",self.warmup_steps)
        print("Minimum LR : ",self.min_learning_rate)
    #-------Training Section Methods--------#
    def __train_batch(self,batch : dict):
        input_ids = batch["input_ids"]
        attention_mask = batch.get("attention_mask")
        labels = batch["labels"]
        optimizer_step = False
        # ---------------------------------------------------------
        # Forward
        # ---------------------------------------------------------
        with self.accelerator.accumulate(self.model):
            with self.accelerator.autocast():
                logits = self.model(input_ids=input_ids,attention_mask=attention_mask)
                loss = self.loss_fn(
                    logits.view(-1, logits.size(-1)),
                    labels.view(-1)
                )

            self.accelerator.backward(loss)
            # Gradient Accumulation
            if self.accelerator.sync_gradients:
                self.accelerator.clip_grad_norm_(
                    self.model.parameters(),
                    self.config.training.max_grad_norm
                )
                # Optimizer Step
                self.optimizer.step()
                self.scheduler.step()

                self.optimizer.zero_grad(set_to_none=True)

                self.global_step += 1
                optimizer_step = True

        return loss.item(),optimizer_step

    def train(self):

        token_seen = 0

        self.model.train()
        for epoch in range(self.epochs,self.config.training.epochs):
            progress = tqdm(
                self.train_loader,
                desc=f"Epoch {epoch + 1}/{self.config.training.epochs}",
            )
            running_loss = 0.0
            num_batches = len(self.train_loader)
            val_loss = None

            for batch_idx, batch in enumerate(progress):

                loss,optimizer_step = self.__train_batch(batch)

                # Validate every N optimizer steps
                if self.val_loader and self.global_step > 0 and optimizer_step and(self.global_step % self.validate_every_step == 0):
                    val_loss  = self.__validate()
                    self.__save_checkpoint(self.checkpoint_save_dir, "latest")

                running_loss += loss
                avg_loss = running_loss / (batch_idx + 1)
                # MLflow Tracking
                mlflow.log_metric(
                    "Train Loss",
                    loss,
                    step=self.global_step
                )
                if val_loss is not None:
                    mlflow.log_metric(
                        "Val Loss",
                        val_loss,
                        step=self.global_step
                    )
                token_seen+= batch["attention_mask"].sum().item()
                progress.set_postfix(
                    loss=f"{avg_loss:.5f}",
                    step=self.global_step,
                    token_seen=token_seen,
                )
            # Epoch summary
            message = (
                f"Epoch {epoch + 1}/{self.config.training.epochs} "
                f"| train_loss: "
                f"{running_loss / num_batches:.4f}"
            )

            if val_loss is not None:
                message += f" | val_loss: {val_loss:.4f}"

            print(message)

    @torch.no_grad()
    def __validate(self) -> float:
        assert self.val_loader is not None

        self.model.eval()

        total_loss = 0.0
        num_batches = 0
        progress = tqdm(self.val_loader,desc="Validation",leave=False)

        for batch in progress:
            input_ids = batch.get("input_ids")
            attention_mask = batch.get("attention_mask")
            labels = batch.get("labels")

            with self.accelerator.autocast():
                logits = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                )
                loss = self.loss_fn(
                    logits.view(-1, logits.size(-1)),
                    labels.view(-1)
                )

            total_loss += loss.item()
            num_batches += 1

            progress.set_postfix(
                loss=f"{total_loss / num_batches:.5f}"
            )

        val_loss = total_loss / num_batches
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.__save_checkpoint(self.checkpoint_save_dir,"best")

        self.model.train()
        return val_loss
    #--------Saving and Loading Methods------#
    def __save_checkpoint(self, path_dir: str,prefix:str):
        path = Path(path_dir) / f"{prefix}.pt"

        checkpoint = {
            "model_state_dict": self.accelerator.unwrap_model(self.model).state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict(),

            "epochs" : self.epochs,
            "global_step" : self.global_step,
            "best_val_loss": self.best_val_loss,
        }
        torch.save(
            checkpoint,
            path
        )
        print(f"Checkpoint saved at {path}")

    def __load_checkpoint(self,path_dir: str,prefix:str):
        path = Path(path_dir) / f"{prefix}.pt"
        checkpoint = torch.load(path, map_location="cpu")

        self.accelerator.unwrap_model(self.model).load_state_dict(checkpoint["model_state_dict"])
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        self.scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

        self.epochs = checkpoint["epochs"]
        self.global_step = checkpoint["global_step"]
        self.best_val_loss = checkpoint.get("best_val_loss",float("inf"))
        print("Checkpoint loaded Successfully...")


In [9]:
def load_partial_weights(
    model: nn.Module,
    checkpoint_path: str,
    device: str = "cpu",
):
    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True,
    )

    # Handle checkpoints like {"model_state_dict": ...}
    if "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        checkpoint = checkpoint["state_dict"]

    model_state = model.state_dict()

    loaded = {}
    skipped = {}

    for name, weight in checkpoint.items():
        if name not in model_state:
            skipped[name] = "parameter not found"
        elif model_state[name].shape != weight.shape:
            skipped[name] = (
                f"shape mismatch: checkpoint {tuple(weight.shape)} "
                f"!= model {tuple(model_state[name].shape)}"
            )
        else:
            loaded[name] = weight

    model_state.update(loaded)
    model.load_state_dict(model_state)

    print(f"Loaded:  {len(loaded)} parameters")
    print(f"Skipped: {len(skipped)} parameters")

    return skipped

In [10]:
print(f"Total Batches: {len(train_loader)}")

Total Batches: 3261


In [11]:
from core.config import Config

configs = Config.from_json(path="exp_1/exp_1_config.json")
configs.model

PeakConfig(model_type='Decoder Transformer', model_name='Peak', vocab_size=4096, ctx_length=512, d_model=256, n_layers=6, n_heads=4, n_kv_heads=4, d_ff=1024)

In [16]:
from core.peakmodel import PeakModel

model = PeakModel(configs.model,configs.training.dropout,is_inference_mode=False)
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=configs,
    checkpoint_save_dir="exp_1/checkpoints",
    # checkpoint_load_dir="exp_1/checkpoints",
)
try:
    load_partial_weights(model,"exp_1/checkpoints/best_s.pt")
except Exception as e:
    print("Error")

Using cuda
Total Steps:  3261
Validate After Every Step:  50
Warmup Steps :  97
Minimum LR :  0.0001
Loaded:  51 parameters
Skipped: 0 parameters


In [17]:
c = 0
for i in model.parameters():
    c+=i.numel()
print(f"Model Has {c / 1_000_000}M Parameters")

Model Has 7.341824M Parameters


In [14]:
# pt_to_safetensors(model ,"weights")

In [30]:
mlflow.set_experiment("peak_exp_1")
with mlflow.start_run(run_name="training"):
    trainer.train()

Epoch 1/2:   0%|          | 0/3261 [00:00<?, ?it/s]

KeyboardInterrupt: 